In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [4]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups,falsePositives,&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying owner: HTOC Org'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,url,lastFalsePositive,indicator
0,5629499555060731,2025-06-16T17:42:54Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T12:14:54Z,3.0,64,RITM0589984,...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.220.101.17
1,5629499536014575,2025-03-20T16:54:11Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T12:14:54Z,5.0,68,"On October 29, 2024, the cybersecurity platfor...",...,"[{'id': 473684, 'name': 'CVE-2023-5044', 'last...","[{'id': 6755399443005580, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,111.90.151.120
2,6755399447112729,2025-05-15T12:40:41Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T11:51:51Z,5.0,80,Executive Summary\n \n\nEclecticIQ analysts as...,...,"[{'id': 493475, 'name': 'CL-STA-0048', 'lastUs...","[{'id': 5629499540002138, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.204.56.106
3,5265112,2025-01-23T19:51:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T09:13:26Z,3.0,57,CISA conducted a hunt on IoC's obtained from o...,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 547740, 'dateAdded': '2025-01-27T13:19...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.42.116.210
4,6755399460008269,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T08:54:58Z,4.0,60,"Details\nIn late June 2025, Iran-aligned hackt...",...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...","[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,103.150.218.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
936,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,3.0,90,ACD R&F processed a malspam campaign with a Ne...,...,"[{'id': 390199, 'name': 'hhs-2024090901', 'las...","[{'id': 455233, 'dateAdded': '2024-09-09T11:22...",NaN,NaN,NaN,https://www.shorturl.at/,NaN,www.shorturl.at/,NaN,www.shorturl.at/
937,4303591,2023-03-03T13:53:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-24T23:25:10Z,3.0,72,NaN,...,"[{'id': 35057, 'name': 'FDA Splunk API', 'last...","[{'id': 148157, 'dateAdded': '2023-03-03T13:52...",NaN,NaN,NaN,NaN,NaN,aka.ms/o0ukef,NaN,aka.ms/o0ukef
938,4476331,2023-11-16T13:43:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:44Z,4.0,85,NaN,...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...","[{'id': 291847, 'dateAdded': '2023-12-15T13:16...",NaN,NaN,NaN,http://selligenttier.naylorcampaigns.com,NaN,selligenttier.naylorcampaigns.com,NaN,selligenttier.naylorcampaigns.com
939,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84,NaN,...,"[{'id': 34822, 'name': 'business email comprom...","[{'id': 149988, 'dateAdded': '2023-04-21T14:08...",NaN,NaN,NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,NaN,geo.netsupportsoftware.com/location/loca.asp


In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"

# Fields to return from GET /v3/indicators/{indicatorId or indicator}
detail_fields = [
    "ownerName", "type", "summary", "dateAdded", "lastModified", "active",
    "confidence", "rating", "threatAssessScore", "observations",
    "tags", "attributes", "securityLabels", "associatedGroups"
]

list_of_indicators = ['207.167.64.24'] 

# Build the keys to fetch
if "observed_src" in globals() and isinstance(observed_src, pd.DataFrame) and not observed_src.empty:
    if "id" in observed_src.columns:
        indicator_keys = (
            observed_src["id"].dropna().astype(str).str.strip().tolist()
        )
    elif "summary" in observed_src.columns:
        indicator_keys = (
            observed_src["summary"].dropna().astype(str).str.strip().tolist()
        )
    else:
        indicator_keys = list_of_indicators
else:
    indicator_keys = list_of_indicators

# De-duplicate while preserving order
seen = set()
ordered_keys = []
for k in indicator_keys:
    if k and k not in seen:
        seen.add(k)
        ordered_keys.append(k)

final_results = []

# Query indicators (single-resource endpoint only)
for key in ordered_keys:
    display(f"Querying indicator: {key}")
    try:
        # URL-encode the path segment (works for both numeric IDs and summary strings)
        path_key = urllib.parse.quote(str(key), safe="")
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{path_key}?fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            result = response.json() or {}
            final_results.append(result)
    except Exception as e:
        pass

# Normalize results
normalized_data = []
for result in final_results:
    data_item = result.get('data', {})
    # v3 single GET usually returns a dict; occasionally data['indicator'] may hold it
    if isinstance(data_item, dict):
        record = data_item.get('indicator', data_item)
        if isinstance(record, dict) and 'summary' in record:
            normalized_data.append(record)
    elif isinstance(data_item, list):
        # Very rare shape; keep parity with earlier normalizer
        for item in data_item:
            if isinstance(item, dict) and 'summary' in item:
                normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    # Derive a simple 'indicator' token from summary (e.g., first token)
    if 'summary' in observed_src.columns:
        observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()

    # Drop dupes on 'indicator' if it exists
    if 'indicator' in observed_src.columns:
        observed_src.drop_duplicates(subset='indicator', inplace=True)

    # Filter to lastObserved >= start, if lastObserved exists
    if 'lastObserved' in observed_src.columns:
        observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
        #observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


In [5]:
# Unnest tags.data in observed_src to get a list of each tag per indicator

# Explode tags.data to one row per tag
tags_exploded = (
    observed_src[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# Extract tag name from each tag object
tags_exploded['tag_name'] = tags_exploded['tags.data'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

# Aggregate all tag names into a list per indicator
tags_per_indicator = (
    tags_exploded.groupby('indicator')['tag_name']
    .apply(lambda x: [t for t in x if t])
    .reset_index()
    .rename(columns={'tag_name': 'tag_list'})
)

# Merge back to observed_src if desired
observed_src_with_tags = observed_src.merge(tags_per_indicator, on='indicator', how='left')

display(observed_src_with_tags[['indicator', 'tag_list']])

,indicator,tag_list
0,185.220.101.17,"[OS Splunk API, FDA Splunk API, CMS Splunk API..."
1,111.90.151.120,"[CVE-2023-5044, CVE-2021-42013, CVE-2021-27850..."
2,15.204.56.106,"[CL-STA-0048, CVE-2025-31324, SAP NetWeaver, U..."
3,192.42.116.210,"[DHA Splunk API, CVE-2025-283, CVE-2025-282, O..."
4,103.150.218.78,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S..."
...,...,...
934,www.shorturl.at/,"[hhs-2024090901, FDA Splunk API, Observed, CDC..."
935,aka.ms/o0ukef,"[FDA Splunk API, Observed, Phishing Email, inv..."
936,selligenttier.naylorcampaigns.com,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp..."
937,geo.netsupportsoftware.com/location/loca.asp,"[business email compromise, hospital, Observed..."


In [6]:
# List of tags to count (case-insensitive, strip whitespace)
tags_of_interest = [
    "Scanning", "DDoS", "Spam", "Phishing", "Cryptojacking",
    "Credential Stuffing", "Ransomware", "Data Theft",
    "Cross Site Scripting Attacks", "SQL Injections"
]

# Normalize tag_list to lowercase for matching
def tag_counter(tag_list, tag):
    if not isinstance(tag_list, list):
        return 0
    return sum(1 for t in tag_list if isinstance(t, str) and t.strip().lower() == tag.lower())

tag_counts = {}
for tag in tags_of_interest:
    tag_counts[tag] = observed_src_with_tags['tag_list'].apply(lambda lst: tag_counter(lst, tag)).sum()
    # Add a column to observed_src with the list of matching "Botnet" tags per indicator
    botnet_tags = set(tags_of_interest)
    def extract_botnet_tags(tag_list):
        if not isinstance(tag_list, list):
            return []
        return [t for t in tag_list if isinstance(t, str) and t.strip().lower() in {b.lower() for b in botnet_tags}]

    # Map indicator to its tag_list for efficient lookup
    indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))

    observed_src['Botnet'] = observed_src['indicator'].map(lambda ind: extract_botnet_tags(indicator_to_tags.get(ind, [])))
# Display as a DataFrame for readability
pd.DataFrame(list(tag_counts.items()), columns=['Tag', 'Count'])

,Tag,Count
0,Scanning,0
1,DDoS,72
2,Spam,0
3,Phishing,18
4,Cryptojacking,0
5,Credential Stuffing,0
6,Ransomware,9
7,Data Theft,8
8,Cross Site Scripting Attacks,0
9,SQL Injections,0


In [7]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        display(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=1)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_26804\1319479530.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


"Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251009.csv']"

'Loaded data from 1 files.'

In [8]:
import pandas as pd

df = observed_src.copy()

#df = df[df['indicator'].isin(observed_data_df['indicator'])]
# --- FULL TAGS UNNEST + PARTNERS ---

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)

# --- GROUPS VIA EXPLODE + GROUPBY ---

#groups_exploded = (
#    df[['indicator', 'associatedGroups.data']]
#      .explode('associatedGroups.data')
#      .dropna(subset=['associatedGroups.data'])
#)

#group_norm = pd.json_normalize(
#    groups_exploded['associatedGroups.data']
#).rename(columns={'id':'group_id','name':'group_name'})

#groups_exploded = groups_exploded.reset_index(drop=True).join(group_norm)

#group_agg = (
#    groups_exploded
#      .drop_duplicates(subset=['indicator','group_id','group_name'])
#      .groupby('indicator', as_index=False)
#      .agg({
#          'group_id':   lambda ids: list(ids),
#          'group_name': lambda names: list(names)
#      })
#      .rename(columns={'group_id':'group_ids','group_name':'group_names'})
#)

# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified', 'falsePositives',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','source','address','url'
]
# Remove 'hostName', 'dnsActive', 'whoisActive' from first_cols to avoid KeyError

# Add 'Botnet' column from observed_src if it exists
if 'Botnet' in observed_src.columns:
    df['Botnet'] = observed_src['Botnet']
    first_cols.append('Botnet')

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c in df.columns]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
#      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)

# apply to all your formerly‑list columns
#for col in ['partners','group_ids','group_names'] + tag_fields:
#    agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)

# Only apply to columns that exist in agg_df to avoid KeyError
for col in ['partners', 'group_ids', 'group_names'] + tag_fields:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)
    
display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,...,tag_id,tag_name,tag_description,tag_lastUsed,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners
0,1.4.195.14,6755399460007413,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:20Z,0,4.0,...,"1469320, 1455870, 505200, 23769, 23576, 98, 12...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...",,"2025-07-18T13:37:52Z, 2025-08-26T18:22:50Z, 20...",,,,,,
1,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T00:38:35Z,0,3.0,...,"471298, 35760, 35057, 30479, 23667, 23630, 235...","DHA Splunk API, OS Splunk API, FDA Splunk API,...",Observations reported by the HHS Ofice of the ...,"2025-10-08T11:06:23Z, 2025-10-07T09:09:13Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS"
2,102.164.252.150,6755399460007416,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T02:53:58Z,0,4.0,...,"1469320, 1455870, 505200, 30479, 23769, 23630,...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...",,"2025-07-18T13:37:52Z, 2025-08-26T18:22:50Z, 20...",,,,,,"CMS, NIH"
3,103.125.173.75,6755399460007958,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:39Z,0,4.0,...,"1469320, 1455870, 505200, 471298, 23769, 23576...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...",,"2025-07-18T13:37:52Z, 2025-08-26T18:22:50Z, 20...",,,,,,DHA
4,103.131.128.10,5629499567010832,2025-09-05T17:42:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-07T04:50:08Z,0,3.0,...,"23769, 23576, 749","VA CSOC CTS Splunk, Observed, malicious",,"2025-10-09T01:31:22Z, 2025-10-09T11:51:51Z, 20...",,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
934,www.deepseek.com,5271608,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-08T04:23:48Z,0,3.0,...,"471298, 461545, 35760, 35752, 35057, 30770, 30...","DHA Splunk API, DeepSeek, OS Splunk API, VA OI...",Observations reported by the HHS Ofice of the ...,"2025-10-08T11:06:23Z, 2025-01-31T14:04:11Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"
935,www.deepseek.com.cdn.cloudflare.net,5271612,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-09T04:37:05Z,0,3.0,...,"471298, 461545, 35752, 30479, 23769, 23630, 23...","DHA Splunk API, DeepSeek, VA OIS CSOC CTS Splu...",,"2025-10-08T11:06:23Z, 2025-01-31T14:04:11Z, 20...",,,,,,"DHA, CMS, NIH, IHS"
936,www.shorturl.at/,4883044,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,0,3.0,...,"390199, 35057, 23576, 23575, 1502, 26","hhs-2024090901, FDA Splunk API, Observed, CDC ...",,"2024-09-10T10:48:34Z, 2025-10-09T11:51:51Z, 20...",,,,,,"FDA, CDC"
937,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-08T19:36:45Z,0,4.0,...,"471298, 35760, 35752, 35057, 30770, 30479, 237...","DHA Splunk API, OS Splunk API, VA OIS CSOC CTS...",Observations reported by the HHS Ofice of the ...,"2025-10-08T11:06:23Z, 2025-10-07T09:09:13Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"


In [9]:
# ──────────────────────────────────────────────────────────────────────────────
# Enrich only final filtered indicators (Shodan & VirusTotalV3) and flatten
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

indicator_values = (
    agg_df["summary"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

display(f"Enriching {len(indicator_values)} indicators with Shodan & VirusTotalV3...")

enriched_results = []
failed = []

for value in indicator_values:
    try:
        encoded_value = urllib.parse.quote(value, safe="")
        q = urllib.parse.urlencode({"type": ["Shodan", "VirusTotalV3"]}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        # Build a fresh RequestObject each loop (adjust to your SDK)
        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        resp.raise_for_status()

        data = resp.json()
        data["summary"] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({"summary": value, "error": str(e)})

# If nothing enriched, just carry on
if not enriched_results:
    display("No enrichment data retrieved.")
    recent_tags = agg_df.copy()

else:
    # One row per summary from enrichment payload
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset="summary", keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on="summary", how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[["summary", COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat["summary"] = exploded["summary"].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            """Flatten one level of lists in a sequence, keep dicts intact."""
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            # if everything is hashable & not dict/list, dedupe
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat  # keep as-is when dicts/lists present

        obj_cols = enrich_flat.select_dtypes("object").columns.difference(["summary"])
        num_cols = enrich_flat.columns.difference(obj_cols.union({"summary"}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        # choose your numeric aggregation; 'max' or 'first'
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby("summary", as_index=False)
              .agg(agg_dict)
        )

        # Remove raw list col and merge flattened cols back
        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset="summary")
                       .merge(enrich_wide, on="summary", how="left")
        )

        display(f"Successfully enriched and merged {len(df_enriched)} indicators.")
    else:
        display("Enrichment column not found; nothing to flatten.")

if failed:
    display(f"{len(failed)} indicators failed to enrich.")


'Enriching 939 indicators with Shodan & VirusTotalV3...'

Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL accentcare.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"More than one indicator matches the criteria you specified. Try looking up by ID instead.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL aka.ms/o0ukef cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host app.hive.co cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host arpdcresources.ca cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Resp

'Successfully enriched and merged 867 indicators.'

'72 indicators failed to enrich.'

In [10]:
recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')

recent_tags

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,1.4.195.14,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:20Z,0,4.0,60,...,[Khueang Nai],[Thailand],"[nt-isp.net, totinternet.net]","[node-d8u.pool-1-4.dynamic.totinternet.net, no...",[TOT Public Company Limited],"[{'transport': 'tcp', 'port': 88, 'product': '...",[TOT Public Company Limited],None,None,1.0
1,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T00:38:35Z,0,3.0,61,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,8.0
2,102.164.252.150,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T02:53:58Z,0,4.0,60,...,[Bata],[Equatorial Guinea],None,None,[Gestora de Infraestructuras de Telecomunicaci...,"[{'transport': 'tcp', 'port': 443, 'ssl': {'is...",[CONEXXIA BATA],None,None,1.0
3,103.125.173.75,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:39Z,0,4.0,60,...,[Medan],[Indonesia],None,None,[PT Trinity Teknologi Nusantara],"[{'transport': 'tcp', 'port': 53}, {'transport...",[PT Trinity Teknologi Nusantara],None,None,1.0
4,103.131.128.10,2025-09-05T17:42:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-07T04:50:08Z,0,3.0,76,...,[Thrissur],[India],[keralavisionisp.com],[keralavisionisp-dynamic-10.128.131.103.kerala...,[Kerala Vision Broad Band Private Limited],"[{'transport': 'tcp', 'port': 80, 'product': '...",[Kerala Vision Broad Band Private Limited],None,None,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
934,www.deepseek.com,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-08T04:23:48Z,0,3.0,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
935,www.deepseek.com.cdn.cloudflare.net,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-09T04:37:05Z,0,3.0,72,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
936,www.shorturl.at/,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,0,3.0,90,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
937,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-08T19:36:45Z,0,4.0,40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
from pymongo import MongoClient

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Threat_Assess_Controlled_Scoring_Data"]
collection = db["controlled_enriched_observation_Data"]

# Convert DataFrame to dictionary records
records = recent_tags.to_dict(orient="records")

# Insert records into MongoDB
result = collection.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} documents into enriched_observation_Data.")

client.close()


In [ ]:
from pymongo import MongoClient
import pandas as pd

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Threat_Assess_Controlled_Scoring_Data"]
collection = db["controlled_enriched_observation_Data"]

# Pull data back from MongoDB
docs = list(collection.find())
recent_tags = pd.DataFrame(docs)
print(f"Pulled {len(recent_tags)} documents from enriched_observation_Data.")


In [11]:
#recent_tags = recent_tags.head(50)
recent_tags.loc[recent_tags['enrich_vtMaliciousCount'].isnull(), 'enrich_vtMaliciousCount'] = 0
display(recent_tags)

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,1.4.195.14,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:20Z,0,4.0,60,...,[Khueang Nai],[Thailand],"[nt-isp.net, totinternet.net]","[node-d8u.pool-1-4.dynamic.totinternet.net, no...",[TOT Public Company Limited],"[{'transport': 'tcp', 'port': 88, 'product': '...",[TOT Public Company Limited],None,None,1.0
1,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T00:38:35Z,0,3.0,61,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,8.0
2,102.164.252.150,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T02:53:58Z,0,4.0,60,...,[Bata],[Equatorial Guinea],None,None,[Gestora de Infraestructuras de Telecomunicaci...,"[{'transport': 'tcp', 'port': 443, 'ssl': {'is...",[CONEXXIA BATA],None,None,1.0
3,103.125.173.75,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:39Z,0,4.0,60,...,[Medan],[Indonesia],None,None,[PT Trinity Teknologi Nusantara],"[{'transport': 'tcp', 'port': 53}, {'transport...",[PT Trinity Teknologi Nusantara],None,None,1.0
4,103.131.128.10,2025-09-05T17:42:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-07T04:50:08Z,0,3.0,76,...,[Thrissur],[India],[keralavisionisp.com],[keralavisionisp-dynamic-10.128.131.103.kerala...,[Kerala Vision Broad Band Private Limited],"[{'transport': 'tcp', 'port': 80, 'product': '...",[Kerala Vision Broad Band Private Limited],None,None,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
934,www.deepseek.com,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-08T04:23:48Z,0,3.0,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
935,www.deepseek.com.cdn.cloudflare.net,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-09T04:37:05Z,0,3.0,72,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
936,www.shorturl.at/,2024-09-09T11:22:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-25T23:24:38Z,0,3.0,90,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
937,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-08T19:36:45Z,0,4.0,40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [ ]:
# Count how many indicators are associated with a unique enrich_domains value

# Check if exploded variable exists and has the required column
if 'exploded' in locals() and 'enrich_domains' in exploded.columns:
    # Drop rows where enrich_domains is missing or NaN
    domains_df = exploded.dropna(subset=['enrich_domains'])

    # If enrich_domains is a list, flatten to individual domain rows
    def flatten_domains(row):
        val = row['enrich_domains']
        if isinstance(val, list):
            return [(row['indicator'], d) for d in val if pd.notna(d)]
        elif pd.notna(val):
            return [(row['indicator'], val)]
        return []

    flat = []
    for _, row in domains_df.iterrows():
        flat.extend(flatten_domains(row))

    flat_df = pd.DataFrame(flat, columns=['indicator', 'domain'])

    # Count unique indicators per domain
    domain_counts = (
        flat_df.groupby('domain')['indicator']
        .nunique()
        .reset_index()
        .rename(columns={'indicator': 'indicator_count'})
        .sort_values('indicator_count', ascending=False)
    )

    display(domain_counts)
# If the required data isn't available, just skip this analysis silently

Error: 'exploded' variable not found or missing 'enrich_domains' column
Available columns in exploded: ['summary', 'data.enrichment.data']


In [14]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=365)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_26804\1921769075.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251009.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251008.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251007.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251006.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251005.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251004.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251003.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251002.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251001.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250930.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250929.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250928.csv', 'Z:/HTOC/Data_Analytics/Data/Op

In [15]:
# Aggregate observed_data_df by 'indicator', counting unique obs_date occurrences
agg_by_indicator = (
    observed_data_df
    .groupby('indicator', as_index=False)['obs_date']
    .nunique()
    .rename(columns={'obs_date': 'obs_days_count'})
)

agg_by_indicator = agg_by_indicator[agg_by_indicator['indicator'].isin(recent_tags['indicator'])]
# Map obs_days_count from agg_by_indicator to recent_tags by indicator, rename to obs_count
recent_tags = recent_tags.merge(
    agg_by_indicator.rename(columns={'obs_days_count': 'obs_count'}),
    on='indicator',
    how='left'
)
display(agg_by_indicator)

,indicator,obs_days_count
4,1.4.195.14,8
18,101.89.174.236,142
24,102.164.252.150,3
80,103.125.173.75,6
82,103.131.128.10,4
...,...,...
5322,www.deepseek.com,235
5323,www.deepseek.com.cdn.cloudflare.net,188
5356,www.shorturl.at/,181
5359,www.sthda.com,248


In [ ]:
import re

# helper to flatten list-or-str into a list of strings
def flatten_hosts(hosts):
    if isinstance(hosts, list):
        return [h for h in hosts if isinstance(h, str)]
    elif isinstance(hosts, str):
        return [hosts]
    return []

# regex: 
# ^[A-Za-z]+       → first label is one “word” (letters only)
# (?:\.[^.]+){3}$  → exactly three more “.” followed by non-dot characters
four_label_word_prefix = re.compile(r'^[A-Za-z]+(?:\.[^.]+){3}$')

# Try to match on 'enrich_org' instead of 'enrich_hostNames'
if 'enrich_org' in recent_tags.columns:
    global matched_results
    exploded = recent_tags.explode('enrich_org')

    mask = exploded['enrich_org'].astype(str).str.match(four_label_word_prefix, na=False)

    matched = recent_tags.loc[exploded[mask].index.unique()]
    matched_results = matched[['indicator','enrich_org']]
else:
    matched = pd.DataFrame(columns=recent_tags.columns)
    matched_results = None


display(matched_results)


In [ ]:
def has_false_positive(tag_str):
    if not isinstance(tag_str, str):
        return False
    tags = [t.strip().lower() for t in tag_str.split(",")]
    return any("false positive" in t for t in tags)

false_positive_indicators = recent_tags[
    recent_tags['tag_name'].apply(has_false_positive)
]

display(false_positive_indicators[['indicator', 'tag_name']])


In [18]:
import pandas as pd
import numpy as np

df = recent_tags.copy()

# ── Severity Binning ────────────────────────────
scoring_bins = [0, 200, 500, 800, 1001]
label_bins = ['low', 'medium', 'high', 'critical']

# ── Feature Weights ─────────────────────────────
Weights = {
    "MALICIOUS_WEIGHT": 2.00,
    "OBSERVATION_COUNT_WEIGHT": 0.02,
    "CONTINUITY_WEIGHT": 0.10,
    "TC_RATING": 0.05,
    "TC_CONFIDENCE": 0.02,
}

# ── Continuity Map ──────────────────────────────
CONTINUITY_MAP = {
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}

BOTNET_ACTIONS = {'scanning','ddos','spam','phishing','cryptojacking','credential stuffing', 'ransomware'}

# Known feature maximums (absolute, domain-informed)
VT_MAX = 94
MAX_OBS_REALISTIC = 365
MAX_RATING = 5
MAX_CONFIDENCE = 100
BOTNET_MULTIPLIER = 0.4
FALSE_POSITIVE_WEIGHT = 0.9  # 10% penalty
SCANNER_PENALTY_MULTIPLIER = 0.7  # 30% penalty

# ── Input caps ──────────────────────────────────
df['obs_count'] = pd.to_numeric(df.get('obs_count', 0), errors='coerce').fillna(0).clip(0, MAX_OBS_REALISTIC)
df['enrich_vtMaliciousCount'] = (
    pd.to_numeric(df.get('enrich_vtMaliciousCount', 0), errors='coerce')
      .fillna(0).clip(0, VT_MAX)
)
df['rating'] = pd.to_numeric(df.get('rating', 0), errors='coerce').fillna(0).clip(0, MAX_RATING)
df['confidence'] = pd.to_numeric(df.get('confidence', 0), errors='coerce').fillna(0).clip(0, MAX_CONFIDENCE)

# ── Raw weighted contributions (no obs additive term) ───────────
df['malicious_scaled'] = np.log1p(df['enrich_vtMaliciousCount'])
df['malicious_raw_score'] = df['malicious_scaled'] * Weights['MALICIOUS_WEIGHT']

df['continuity_val'] = df.get('type', '').map(CONTINUITY_MAP).fillna(0)
df['continuity_raw_score'] = df['continuity_val'] * Weights['CONTINUITY_WEIGHT']

df['tc_raw_rating'] = df['rating'] * Weights['TC_RATING']
df['tc_raw_confidence'] = df['confidence'] * Weights['TC_CONFIDENCE']

df['raw_score'] = (
    df['malicious_raw_score'] +
    df['continuity_raw_score'] +
    df['tc_raw_rating'] +
    df['tc_raw_confidence']
)

# ── Observation penalty (multiplier) ────────────
OBS_PENALTY_STRENGTH = float(Weights['OBSERVATION_COUNT_WEIGHT'])
OBS_MIN_MULTIPLIER = 0.50  # floor so max penalty is 50% reduction (tune as you like)

obs_frac = df['obs_count'] / MAX_OBS_REALISTIC
df['obs_penalty_multiplier'] = (1.0 - OBS_PENALTY_STRENGTH * obs_frac).clip(OBS_MIN_MULTIPLIER, 1.0)

# Apply obs penalty to the base raw score
df['raw_score'] = df['raw_score'] * df['obs_penalty_multiplier']

# ── Botnet flag (robust) ────────────────────────
col = df.get('Botnet', None)
if col is None:
    df['botnet_flag'] = 0
else:
    def is_botnet(val):
        text = " ".join(map(str, val)).lower() if isinstance(val, (list, set, tuple)) else str(val).lower()
        return int(any(action in text for action in BOTNET_ACTIONS))
    df['botnet_flag'] = pd.Series(col).apply(is_botnet).astype(int)

# ── False Positive penalty (preview + conditional apply) ────────
if 'falsePositives' in df.columns:
    FALSE_POSITIVE_MULTIPLIER = FALSE_POSITIVE_WEIGHT
    df['falsePositives'] = pd.to_numeric(df['falsePositives'], errors='coerce').fillna(0)
    mask_fp = df['falsePositives'] > 0

    # Always show what the FP-applied score would be
    df['false_positive_raw_score'] = df['raw_score'] * FALSE_POSITIVE_MULTIPLIER

    # Apply FP only where present
    df.loc[mask_fp, 'raw_score'] = df.loc[mask_fp, 'false_positive_raw_score']
else:
    df['false_positive_raw_score'] = df['raw_score']

# ── Botnet penalty application ──────────────────
df['raw_score'] = np.where(df['botnet_flag'] == 1, df['raw_score'] * BOTNET_MULTIPLIER, df['raw_score'])

# ── Scanner penalty ───────────────────────────────
def has_scanner_tag(tags):
    if isinstance(tags, list):
        return any(str(t).strip().lower() == 'scanner' for t in tags)
    if isinstance(tags, str):
        return 'scanner' in [t.strip().lower() for t in tags.split(',')]
    return False

scanner_mask = df['enrich_tags'].apply(has_scanner_tag)
df.loc[scanner_mask, 'raw_score'] = df.loc[scanner_mask, 'raw_score'] * SCANNER_PENALTY_MULTIPLIER

# ── Absolute Cap (remove obs term since it's a multiplier now) ──
RAW_SCORE_CAP = (
    np.log1p(VT_MAX) * Weights['MALICIOUS_WEIGHT'] +
    max(CONTINUITY_MAP.values()) * Weights['CONTINUITY_WEIGHT'] +
    (MAX_RATING * Weights['TC_RATING']) +
    (MAX_CONFIDENCE * Weights['TC_CONFIDENCE'])
)

# ── Normalize to 0–1000 ────────────────────────
df['Threat_Score'] = (1000 * (df['raw_score'] / RAW_SCORE_CAP).clip(0, 1)).round().fillna(0).astype(int)

# ── Assign Severity bin ─────────────────────────
df['Severity'] = pd.cut(df['Threat_Score'], bins=scoring_bins, labels=label_bins, right=False)

# ── Explanation (now reports obs penalty) ───────
_NAME_MAP = {
    'malicious_raw_score': 'VT malicious (log-scaled)',
    'continuity_raw_score': 'Continuity (indicator type)',
    'tc_raw_rating': 'TC rating',
    'tc_raw_confidence': 'TC confidence',
}

def build_explanation(row):
    parts = {
        'malicious_raw_score': float(row.get('malicious_raw_score', 0) or 0),
        'continuity_raw_score': float(row.get('continuity_raw_score', 0) or 0),
        'tc_raw_rating': float(row.get('tc_raw_rating', 0) or 0),
        'tc_raw_confidence': float(row.get('tc_raw_confidence', 0) or 0),
    }
    total = float(row.get('raw_score', 0) or 0)
    score = float(row.get('Threat_Score', 0) or 0)
    sev = str(row.get('Severity', 'nan'))

    # Top additive drivers
    contrib_items = sorted(parts.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
    driver_bits = []
    for k, v in contrib_items:
        label = _NAME_MAP.get(k, k)
        if total != 0:
            pct = 100.0 * (v / total)
            driver_bits.append(f"{label} ({v:+.2f}, {pct:+.1f}% of total)")
        else:
            driver_bits.append(f"{label} ({v:+.2f})")

    # Penalty notes
    obs_mult = float(row.get('obs_penalty_multiplier', 1.0) or 1.0)
    obs_note = f"Observations penalty: ×{obs_mult:.2f} ({(1-obs_mult)*100:.1f}% reduction)."

    botnet_flag = int(row.get('botnet_flag', 0) or 0)
    botnet_note = "Botnet penalty applied (60%)." if botnet_flag == 1 else "No botnet penalty."

    fp_cnt = int(row.get('falsePositives', 0) or 0)
    fp_note = f"{fp_cnt} false positive tag(s): 10% penalty applied." if fp_cnt > 0 else "No false positive tags."

    cap_text = f"Raw score uses {100.0 * (total / RAW_SCORE_CAP):.1f}% of cap; normalized to {score:.0f}/1000." if RAW_SCORE_CAP > 0 else f"Normalized score {score:.0f}/1000."

    scanner_penalty = float(row.get('raw_score', 0)) < float(row.get('false_positive_raw_score', 0))
    scanner_note = "Scanner penalty applied (30%)." if scanner_penalty else "No scanner penalty."

    # Add scanner_note to the explanation
    return (
        f"Severity: {sev}. Top drivers: " + "; ".join(driver_bits) + ". "
        f"{obs_note} {botnet_note} {fp_note} {cap_text} {scanner_note}"
    )

df['Explanation'] = df.apply(build_explanation, axis=1)

# drop duplicates
df.drop_duplicates(subset='indicator', inplace=True)

display(df[['indicator','type','enrich_vtMaliciousCount','obs_count',
            'malicious_raw_score','continuity_raw_score','rating','tc_raw_rating',
            'tc_raw_confidence','obs_penalty_multiplier','botnet_flag',
            'false_positive_raw_score','falsePositives',
            'raw_score','Threat_Score','Severity','Explanation']])


,indicator,type,enrich_vtMaliciousCount,obs_count,malicious_raw_score,continuity_raw_score,rating,tc_raw_rating,tc_raw_confidence,obs_penalty_multiplier,botnet_flag,false_positive_raw_score,falsePositives,raw_score,Threat_Score,Severity,Explanation
0,1.4.195.14,Address,1.0,8,1.386294,0.1,4.0,0.20,1.20,0.999562,1,2.596526,0,1.154012,99,low,Severity: low. Top drivers: VT malicious (log-...
1,101.89.174.236,Address,8.0,142,4.394449,0.1,3.0,0.15,1.22,0.992219,0,5.236937,0,5.818819,499,medium,Severity: medium. Top drivers: VT malicious (l...
2,102.164.252.150,Address,1.0,3,1.386294,0.1,4.0,0.20,1.20,0.999836,1,2.597238,0,1.154328,99,low,Severity: low. Top drivers: VT malicious (log-...
3,103.125.173.75,Address,1.0,6,1.386294,0.1,4.0,0.20,1.20,0.999671,1,2.596811,0,1.154138,99,low,Severity: low. Top drivers: VT malicious (log-...
4,103.131.128.10,Address,1.0,4,1.386294,0.1,3.0,0.15,1.52,0.999781,0,2.840042,0,3.155603,271,medium,Severity: medium. Top drivers: TC confidence (...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
934,www.deepseek.com,Host,0.0,235,0.000000,0.2,3.0,0.15,1.48,0.987123,0,1.625792,0,1.806436,155,low,Severity: low. Top drivers: TC confidence (+1....
935,www.deepseek.com.cdn.cloudflare.net,Host,0.0,188,0.000000,0.2,3.0,0.15,1.44,0.989699,0,1.594404,0,1.771561,152,low,Severity: low. Top drivers: TC confidence (+1....
936,www.shorturl.at/,Stripped URL,0.0,181,0.000000,0.0,3.0,0.15,1.80,0.990082,0,1.737594,0,1.930660,166,low,Severity: low. Top drivers: TC confidence (+1....
937,www.sthda.com,Host,0.0,248,0.000000,0.2,4.0,0.20,0.80,0.986411,0,1.065324,0,1.183693,102,low,Severity: low. Top drivers: TC confidence (+0....


In [ ]:
import pandas as pd
import numpy as np

df = recent_tags.copy()

# ── Severity Binning ────────────────────────────
scoring_bins = [0, 200, 500, 800, 1001]
label_bins = ['low', 'medium', 'high', 'critical']

# ── Feature Weights / Params ────────────────────
Weights = {
    "MALICIOUS_WEIGHT": 2.00,
    "OBSERVATION_COUNT_WEIGHT": 0.02,
    "CONTINUITY_WEIGHT": 0.10,
    "TC_RATING": 0.05,
    "TC_CONFIDENCE": 0.02,
}

BOTNET_ACTIONS = {
    'scanning','ddos','spam','phishing','cryptojacking','credential stuffing','ransomware'
}

# Known feature maximums (absolute, domain-informed)
VT_MAX = 94
MAX_OBS_REALISTIC = 365
MAX_RATING = 5
MAX_CONFIDENCE = 100

# Policy multipliers
FALSE_POSITIVE_WEIGHT = 0.9         # 10% penalty
BOTNET_MULTIPLIER     = 0.4         # 60% penalty when botnet-related
SCANNER_PENALTY_MULTIPLIER = 0.7    # 30% penalty for scanner/benign-crawler tags
DATA_QUALITY_FLOOR    = 0.85        # at worst 15% reduction for poor completeness

# ── Input caps ──────────────────────────────────
df['obs_count'] = pd.to_numeric(df.get('obs_count', 0), errors='coerce').fillna(0).clip(0, MAX_OBS_REALISTIC)
df['enrich_vtMaliciousCount'] = (
    pd.to_numeric(df.get('enrich_vtMaliciousCount', 0), errors='coerce')
      .fillna(0).clip(0, VT_MAX)
)
df['rating']     = pd.to_numeric(df.get('rating', 0), errors='coerce').fillna(0).clip(0, MAX_RATING)
df['confidence'] = pd.to_numeric(df.get('confidence', 0), errors='coerce').fillna(0).clip(0, MAX_CONFIDENCE)
df['type'] = df.get('type', '').astype(str)

# ── Base additive evidence (no obs additive term) ───────────────
df['malicious_scaled']     = np.log1p(df['enrich_vtMaliciousCount'])
df['malicious_raw_score']  = df['malicious_scaled'] * Weights['MALICIOUS_WEIGHT']
df['continuity_val']       = df['type'].map({
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}).fillna(0)
df['continuity_raw_score'] = df['continuity_val'] * Weights['CONTINUITY_WEIGHT']
df['tc_raw_rating']        = df['rating']     * Weights['TC_RATING']
df['tc_raw_confidence']    = df['confidence'] * Weights['TC_CONFIDENCE']

df['raw_score'] = (
    df['malicious_raw_score'] +
    df['continuity_raw_score'] +
    df['tc_raw_rating'] +
    df['tc_raw_confidence']
)

# ── Observation penalty (multiplier; linear) ────────────────────
OBS_PENALTY_STRENGTH = float(Weights['OBSERVATION_COUNT_WEIGHT'])
OBS_MIN_MULTIPLIER   = 0.50  # floor so max reduction is 50% (tune to taste)

obs_frac = df['obs_count'] / MAX_OBS_REALISTIC
df['obs_penalty_multiplier'] = (1.0 - OBS_PENALTY_STRENGTH * obs_frac).clip(OBS_MIN_MULTIPLIER, 1.0)
df['raw_score'] *= df['obs_penalty_multiplier']

# ── Data quality multiplier (light guard) ───────────────────────
needed = ['type','rating','confidence','enrich_vtMaliciousCount']
present_frac = df[needed].notna().sum(axis=1) / len(needed)
df['data_quality_multiplier'] = present_frac.clip(DATA_QUALITY_FLOOR, 1.0)
df['raw_score'] *= df['data_quality_multiplier']

# ── Botnet flag (robust, multi-word, list/string) ───────────────
col = df.get('Botnet', None)
if col is None:
    df['botnet_flag'] = 0
else:
    def is_botnet(val):
        text = " ".join(map(str, val)).lower() if isinstance(val, (list, set, tuple)) else str(val).lower()
        return int(any(action in text for action in BOTNET_ACTIONS))
    df['botnet_flag'] = pd.Series(col).apply(is_botnet).astype(int)

# ── False Positive penalty (preview + conditional apply) ────────
if 'falsePositives' in df.columns:
    df['falsePositives'] = pd.to_numeric(df['falsePositives'], errors='coerce').fillna(0)
    mask_fp = df['falsePositives'] > 0
    # Preview: always "what if" penalized
    df['false_positive_raw_score'] = df['raw_score'] * FALSE_POSITIVE_WEIGHT
    # Apply FP only where present
    df.loc[mask_fp, 'raw_score'] = df.loc[mask_fp, 'false_positive_raw_score']
else:
    df['false_positive_raw_score'] = df['raw_score']

# ── Botnet penalty application ──────────────────────────────────
df['botnet_penalty_multiplier'] = np.where(df['botnet_flag'] == 1, BOTNET_MULTIPLIER, 1.0)
df['raw_score'] *= df['botnet_penalty_multiplier']

# ── Scanner penalty (robust tag parse) ──────────────────────────
def has_scanner_tag(val):
    scanners = {'scanner','masscan','zmap','shodan','censys'}
    if isinstance(val, (list, set, tuple)):
        t = " ".join(map(str, val)).lower()
    elif isinstance(val, str):
        # supports csv-ish strings
        t = " ".join(x.strip() for x in val.split(',')).lower()
    else:
        t = str(val).lower()
    return any(s in t for s in scanners)

if 'enrich_tags' in df.columns:
    scanner_mask = df['enrich_tags'].apply(has_scanner_tag)
else:
    scanner_mask = pd.Series(False, index=df.index)

df['scanner_penalty_multiplier'] = np.where(scanner_mask, SCANNER_PENALTY_MULTIPLIER, 1.0)
df['raw_score'] *= df['scanner_penalty_multiplier']

# ── Absolute Cap (multipliers are NOT in cap; only additive parts) ───────────
RAW_SCORE_CAP = (
    np.log1p(VT_MAX) * Weights['MALICIOUS_WEIGHT'] +
    float(df['continuity_val'].max() if len(df) else 3) * Weights['CONTINUITY_WEIGHT'] +
    (MAX_RATING * Weights['TC_RATING']) +
    (MAX_CONFIDENCE * Weights['TC_CONFIDENCE'])
)

# ── Normalize to 0–1000 ─────────────────────────────────────────
df['Threat_Score'] = (1000 * (df['raw_score'] / RAW_SCORE_CAP).clip(0, 1)).round().fillna(0).astype(int)

# ── Assign Severity bin ─────────────────────────────────────────
df['Severity'] = pd.cut(df['Threat_Score'], bins=scoring_bins, labels=label_bins, right=False)

# ── Explanation (drivers + multipliers) ─────────────────────────
_NAME_MAP = {
    'malicious_raw_score': 'VT malicious (log-scaled)',
    'continuity_raw_score': 'Continuity (indicator type)',
    'tc_raw_rating': 'TC rating',
    'tc_raw_confidence': 'TC confidence',
}

def build_explanation(row):
    parts = {
        'malicious_raw_score': float(row.get('malicious_raw_score', 0) or 0),
        'continuity_raw_score': float(row.get('continuity_raw_score', 0) or 0),
        'tc_raw_rating': float(row.get('tc_raw_rating', 0) or 0),
        'tc_raw_confidence': float(row.get('tc_raw_confidence', 0) or 0),
    }
    total = float(row.get('raw_score', 0) or 0)
    score = float(row.get('Threat_Score', 0) or 0)
    sev = str(row.get('Severity', 'nan'))

    # Top additive drivers
    contrib = sorted(parts.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
    driver_bits = []
    for k, v in contrib:
        label = _NAME_MAP.get(k, k)
        if total != 0:
            pct = 100.0 * (v / total)
            driver_bits.append(f"{label} ({v:+.2f}, {pct:+.1f}% of total)")
        else:
            driver_bits.append(f"{label} ({v:+.2f})")

    # Multipliers
    obs_mult     = float(row.get('obs_penalty_multiplier', 1.0))
    dq_mult      = float(row.get('data_quality_multiplier', 1.0))
    botnet_mult  = float(row.get('botnet_penalty_multiplier', 1.0))
    scanner_mult = float(row.get('scanner_penalty_multiplier', 1.0))
    fp_cnt       = int(row.get('falsePositives', 0) or 0)

    obs_note    = f"Observations ×{obs_mult:.2f} ({(1-obs_mult)*100:.1f}% reduction)."
    dq_note     = f"Data quality ×{dq_mult:.2f}."
    botnet_note = "Botnet ×0.40 applied." if botnet_mult < 1.0 else "No botnet penalty."
    fp_note     = f"{fp_cnt} false positive tag(s): ×0.90 applied." if fp_cnt > 0 else "No false positive tags."
    scan_note   = "Scanner ×0.70 applied." if scanner_mult < 1.0 else "No scanner penalty."

    cap_text = f"Raw score uses {100.0 * (total / RAW_SCORE_CAP):.1f}% of cap; normalized to {score:.0f}/1000." if RAW_SCORE_CAP > 0 else f"Normalized score {score:.0f}/1000."

    return (
        f"Severity: {sev}. Top drivers: " + "; ".join(driver_bits) + ". "
        f"{obs_note} {dq_note} {botnet_note} {fp_note} {scan_note} {cap_text}"
    )

df['Explanation'] = df.apply(build_explanation, axis=1)

# ── De-dupe and show ────────────────────────────────────────────
if 'indicator' in df.columns:
    df.drop_duplicates(subset='indicator', inplace=True)

display(df[['indicator','type','enrich_vtMaliciousCount','obs_count',
            'rating','botnet_flag','falsePositives',
            'raw_score','Threat_Score','Severity','Explanation']])

#how many partners seen an indicator. If low value, but seen across multiple partners, flag as suspicious

,indicator,type,enrich_vtMaliciousCount,obs_count,rating,botnet_flag,falsePositives,raw_score,Threat_Score,Severity,Explanation
0,1.4.195.14,Address,1.0,8,4.0,1,0,1.154012,100,low,Severity: low. Top drivers: VT malicious (log-...
1,101.89.174.236,Address,8.0,142,3.0,0,0,5.818819,503,high,Severity: high. Top drivers: VT malicious (log...
2,102.164.252.150,Address,1.0,3,4.0,1,0,1.154328,100,low,Severity: low. Top drivers: VT malicious (log-...
3,103.125.173.75,Address,1.0,6,4.0,1,0,1.154138,100,low,Severity: low. Top drivers: VT malicious (log-...
4,103.131.128.10,Address,1.0,4,3.0,0,0,3.155603,273,medium,Severity: medium. Top drivers: TC confidence (...
...,...,...,...,...,...,...,...,...,...,...,...
934,www.deepseek.com,Host,0.0,235,3.0,0,0,1.806436,156,low,Severity: low. Top drivers: TC confidence (+1....
935,www.deepseek.com.cdn.cloudflare.net,Host,0.0,188,3.0,0,0,1.771561,153,low,Severity: low. Top drivers: TC confidence (+1....
936,www.shorturl.at/,Stripped URL,0.0,181,3.0,0,0,1.930660,167,low,Severity: low. Top drivers: TC confidence (+1....
937,www.sthda.com,Host,0.0,248,4.0,0,0,1.183693,102,low,Severity: low. Top drivers: TC confidence (+0....


In [ ]:
# Calculate the ratio of each Severity bin in df
severity_ratio = (df['Severity'].value_counts(normalize=True) * 100).round(2).astype(str) + '%'
print(severity_ratio)